In [22]:
import os
import torch
from torchvision import transforms
from PIL import Image
from huggingface_hub import login
from transformers import AutoModel
import general_fcns as gf


In [ ]:
login(token="")

In [16]:
titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True)
conch, eval_transform = titan.return_conch()
conch = conch.eval()

In [ ]:
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.svs', '.tif', '.ndpi')

In [ ]:

def extract_features_from_csv(csv_path, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    # Read the CSV file to get the list of image paths
    try:
        df = pd.read_csv(csv_path)
        image_paths = df['image_path'].tolist()  # Assuming the CSV has a column named 'image_path'
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return
    
    # Iterate over the image paths
    for img_path in image_paths:
        if os.path.exists(img_path) and img_path.lower().endswith(valid_exts):
            filename = os.path.basename(img_path)
            print(f"Processing: {filename}")
            
            try:
                if img_path.lower().endswith('.svs', '.tif', '.ndpi'):
                    # Use gf to extract a tile or scaled image at a specific magnification
                    slide_array = gf.slide_at_magnification(img_path, magnification_params={'magnification': 10})
                    image = Image.fromarray(slide_array)
                else:
                    image = Image.open(img_path).convert("RGB")
                
                img_tensor = eval_transform(image).unsqueeze(0)  # [1, C, H, W]

                with torch.no_grad():
                    features = conch(img_tensor)

                output_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.pt")
                torch.save(features.cpu(), output_path)
                print(f"Saved features for {filename}")
            
            except Exception as e:
                print(f"Error processing {filename}: {e}")
        else:
            print(f"Skipping invalid or non-existent file: {img_path}")

In [ ]:
csv_path = "./filtered_metadata_full_path.csv" 
output_folder = "./features"  


extract_features_from_csv(csv_path, output_folder)

Processing: DIG_PAT_1697003050.svs
Saved features for DIG_PAT_1697003050.svs
Processing: DIG_PAT_1697003063.svs
Saved features for DIG_PAT_1697003063.svs
Processing: DIG_PAT_1697003072.svs
Saved features for DIG_PAT_1697003072.svs
Processing: DIG_PAT_1697003099.svs
Saved features for DIG_PAT_1697003099.svs
Processing: DIG_PAT_1697003125.svs
Saved features for DIG_PAT_1697003125.svs
Processing: DIG_PAT_1697003181.svs
Saved features for DIG_PAT_1697003181.svs
Processing: DIG_PAT_1697003266.svs
Saved features for DIG_PAT_1697003266.svs
Processing: DIG_PAT_1697003309.svs
Saved features for DIG_PAT_1697003309.svs
